# 06 - TF-IDF Features + Multinomial Logistic Regression

This notebook trains a 3-class softmax regression classifier using TF-IDF features
on the cleaned and stratified train/val/test split generated by notebook 05.


In [1]:
import json
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.preprocessing import LabelEncoder

RANDOM_STATE = 42

SPLITS_DIR = Path("data/splits")
MODEL_DIR = Path("models/tfidf")
PLOTS_DIR = Path("plots/models")

MODEL_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)


In [2]:
train_df = pd.read_csv(SPLITS_DIR / "train_papers.csv")
val_df = pd.read_csv(SPLITS_DIR / "val_papers.csv")
test_df = pd.read_csv(SPLITS_DIR / "test_papers.csv")

required_cols = {"model_text", "label"}
for name, frame in [("train", train_df), ("val", val_df), ("test", test_df)]:
    missing = required_cols - set(frame.columns)
    if missing:
        raise ValueError(f"{name} split missing columns: {missing}")

print("Split shapes:", train_df.shape, val_df.shape, test_df.shape)
print("Train label counts:")
print(train_df["label"].value_counts())


Split shapes: (1998, 19) (428, 19) (429, 19)
Train label counts:
label
THEORETICAL    699
APPLIED        698
SURVEY         601
Name: count, dtype: int64


In [3]:
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(train_df["label"])
y_val = label_encoder.transform(val_df["label"])

vectorizer_params = dict(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.95,
    sublinear_tf=True,
    strip_accents="unicode",
)

vectorizer = TfidfVectorizer(**vectorizer_params)
X_train = vectorizer.fit_transform(train_df["model_text"].fillna(""))
X_val = vectorizer.transform(val_df["model_text"].fillna(""))

candidate_cs = [0.25, 0.5, 1.0, 2.0, 4.0]
val_results = []

for c_val in candidate_cs:
    model = LogisticRegression(
        C=c_val,
        max_iter=4000,
        random_state=RANDOM_STATE,
    )
    model.fit(X_train, y_train)
    val_pred = model.predict(X_val)
    macro_f1 = f1_score(y_val, val_pred, average="macro")
    val_results.append({"C": c_val, "val_macro_f1": float(macro_f1)})

val_results_df = pd.DataFrame(val_results).sort_values("val_macro_f1", ascending=False)
best_c = float(val_results_df.iloc[0]["C"])

print("Validation results:")
print(val_results_df)
print(f"Selected C: {best_c}")

Validation results:
      C  val_macro_f1
4  4.00      0.972579
3  2.00      0.967997
2  1.00      0.967997
1  0.50      0.963341
0  0.25      0.956542
Selected C: 4.0


In [4]:
train_val_df = pd.concat([train_df, val_df], ignore_index=True)
y_train_val = label_encoder.transform(train_val_df["label"])
y_test = label_encoder.transform(test_df["label"])

final_vectorizer = TfidfVectorizer(**vectorizer_params)
X_train_val = final_vectorizer.fit_transform(train_val_df["model_text"].fillna(""))
X_test = final_vectorizer.transform(test_df["model_text"].fillna(""))

final_model = LogisticRegression(
    C=best_c,
    max_iter=4000,
    random_state=RANDOM_STATE,
)
final_model.fit(X_train_val, y_train_val)
y_test_pred = final_model.predict(X_test)

report = classification_report(
    y_test,
    y_test_pred,
    target_names=label_encoder.classes_,
    output_dict=True,
    digits=4,
)

metrics = {
    "model": "tfidf_logreg_multinomial",
    "best_c": best_c,
    "accuracy": float(accuracy_score(y_test, y_test_pred)),
    "macro_f1": float(f1_score(y_test, y_test_pred, average="macro")),
    "weighted_f1": float(f1_score(y_test, y_test_pred, average="weighted")),
    "labels": list(label_encoder.classes_),
}

report_df = pd.DataFrame(report).transpose()
report_df

,precision,recall,f1-score,support
APPLIED,0.967105,0.980000,0.973510,150.000000
SURVEY,0.984496,0.984496,0.984496,129.000000
THEORETICAL,0.993243,0.980000,0.986577,150.000000
accuracy,0.981352,0.981352,0.981352,0.981352
macro avg,0.981615,0.981499,0.981528,429.000000
weighted avg,0.981474,0.981352,0.981382,429.000000


In [5]:
joblib.dump(final_vectorizer, MODEL_DIR / "tfidf_vectorizer.joblib")
joblib.dump(final_model, MODEL_DIR / "tfidf_logreg.joblib")
joblib.dump(label_encoder, MODEL_DIR / "label_encoder.joblib")

with open(MODEL_DIR / "tfidf_metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

val_results_df.to_csv(MODEL_DIR / "tfidf_val_results.csv", index=False)
report_df.to_csv(MODEL_DIR / "tfidf_classification_report.csv", index=True)

cm = confusion_matrix(y_test, y_test_pred)
fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_encoder.classes_)
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("TF-IDF + Multinomial Logistic Regression")
fig.tight_layout()
fig.savefig(PLOTS_DIR / "tfidf_confusion_matrix.png", dpi=160)
plt.close(fig)

print("Saved TF-IDF artifacts to models/tfidf and plots/models.")
metrics


Saved TF-IDF artifacts to models/tfidf and plots/models.


{'model': 'tfidf_logreg_multinomial',
 'best_c': 4.0,
 'accuracy': 0.9813519813519813,
 'macro_f1': 0.9815277463379651,
 'weighted_f1': 0.981382441136208,
 'labels': ['APPLIED', 'SURVEY', 'THEORETICAL']}